# Work Session 4: Overfitting, data leakage and generalization

This notebook organizes the solution by assignment part. Run the first cell from this folder to regenerate the results and plots. Detailed answers are also saved as separate Markdown files.

In [ ]:
%run ./solution.py

## Part A: inspection
- Input row order chronological: False; exact duplicates: 0; excluding student_id: 0; ignoring timestamp: 0.
- Cohort categories with fewer than 10 rows: 12 of 42.
- Missing values (columns with any): daily_productivity (34, 2.8%), revision_frequency (34, 2.8%), focus_duration (43, 3.6%), screen_time_non_study (32, 2.7%), energy_level (28, 2.3%), stress_level (35, 2.9%), routine_rating (41, 3.4%), sleep_hours (55, 4.6%), online_courses (36, 3.0%), programming_foundation (41, 3.4%), events_participation (58, 4.8%), external_resources (45, 3.8%), external_pressure (29, 2.4%).
- Numerical association with target, strongest positive: attendance_rate (0.525); weakest magnitude: sleep_hours (-0.010).
- Risk label vs CGPA category cross-tab:

cgpa_category           5.0 – 6.9  7.0 – 8.4  8.5 – 9.4  9.5 – 10.0
performance_risk_level                                             
High Risk                     188        221         39           4
Low Risk                        0          4         89          83
Moderate Risk                  48        195        234          95

The risk label must not be used: it is a derived label closely tied to the same self-reported academic outcome and can directly reveal target information. The CGPA time plot is in part_a_cgpa_time.png; its 7-day mean stays broadly stable near 8.0, with only a modest early-April rise and no abrupt drift.

Risks: (1) target-derived performance_risk_level can leak CGPA; (2) duplicates can place the same response in both partitions; (3) random splits can mix earlier and later survey periods; (4) rare cohort categories encourage memorization; (5) flexible trees/polynomials can fit noise in a small, noisy survey.

In [ ]:
from IPython.display import display, Image
display(Image(filename='figures/part_a_cgpa_time.png'))

## Part B: evaluation and leakage
| Strategy | Train RMSE | Test RMSE |
|---|---:|---:|---|
| B1 random split | 0.0000 | 1.5579 |
| B1 random split baseline (training mean) | 1.2671 | 1.2426 |
| B1 chronological split | 0.0000 | 2.0676 |
| B1 chronological split baseline (training mean) | 1.2685 | 1.2362 |
| B2 duplicates retained | 0.0000 | 1.5906 |
| B2 duplicate rows removed | 0.0000 | 1.7591 |
| B3 scaler fit on all rows | 1.1001 | 1.1858 |
| B3 scaler inside training pipeline | 1.0931 | 1.1904 |

Random splitting mixes dates, so it can hide temporal distribution shift and usually gives an optimistic estimate for deployment on later submissions. In B.2, 180 copies were injected and 47 answer-identical records cross the chronological boundary when timestamps are ignored. The retained-duplicate evaluation can reuse answers across train and test, making test performance optimistic; deduplication before splitting removes these repeated responses. In B.3, whole-data scaling lets the test distribution influence learned means and variances; the gap can be small, but imputation and scaling must be fitted on training data to keep evaluation honest.

## Part C: diagnosing overfitting
| Tree depth | Train RMSE | Test RMSE | Gap |
|---|---:|---:|---:|---|
| 1 | 1.2231 | 1.1744 | -0.0487 |
| 2 | 1.1849 | 1.1458 | -0.0391 |
| 3 | 1.1553 | 1.1341 | -0.0213 |
| 5 | 1.0760 | 1.1523 | 0.0763 |
| 10 | 0.6975 | 1.9967 | 1.2992 |
| 20 | 0.0516 | 2.0559 | 2.0044 |
| None | 0.0000 | 2.0676 | 2.0676 |

The tree train error falls as depth increases, while test error is minimized at depth 3; overfitting becomes evident as the gap expands and test error stops improving. See part_c_tree_depth.png.

| Polynomial degree | 5-fold temporal CV RMSE |
|---:|---:|
| 1 | 1.1850 |
| 2 | 1.1849 |
| 3 | 1.1861 |
| 5 | 1.1925 |
| 9 | 1.2001 |
| 15 | 1.2001 |

Polynomial degree changes a one-dimensional smooth feature representation, so its effect is constrained compared with a tree that partitions a dozen features and can isolate individual observations. See part_c_poly_degree.png.

In [ ]:
from IPython.display import display, Image
display(Image(filename='figures/part_c_tree_depth.png'))

In [ ]:
from IPython.display import display, Image
display(Image(filename='figures/part_c_poly_degree.png'))

## Part D: generalization improvements
### D.1 Cohort encoding (5-fold time-series validation)
| Treatment | CV RMSE |
|---|---:|
| remove | 1.3705 |
| rare_group | 1.3719 |
| target_encode | 1.3640 |

The lowest validation RMSE indicates the preferred treatment. A small improvement over removal means cohort_code adds little reliable predictive information; target encoding is cross-fitted to avoid using each row's target in its own encoding.

### D.2 Submission day
| Model | Feature choice | CV RMSE |
|---|---|---:|
| Linear Regression | with day | 1.1688 |
| Linear Regression | without day | 1.1629 |
| Decision Tree depth 5 | with day | 1.3705 |
| Decision Tree depth 5 | without day | 1.3106 |
| Gradient Boosting | with day | 1.1790 |
| Gradient Boosting | without day | 1.1600 |

All three models had lower validation RMSE without submission_day, so it is excluded from the final feature matrix. Since all future test dates lie past the training range, a linear day effect can extrapolate poorly and tree models cannot extrapolate the trend.

### D.3 Regularization
| Model | Selected alpha | CV RMSE | Lasso zero coefficients |
|---|---:|---:|---:|
| Linear | none | 1.1925 | - |
| Ridge | 10 | 1.1855 | - |
| Lasso | 0.01 | 1.1848 | 5 |

Ridge shrinks all polynomial coefficients continuously; Lasso can set coefficients exactly to zero. The unregularized degree-5 model is more sensitive to noisy curvature.

### D.4 Final untouched test evaluation
Feature set for final comparison: study_hours, sleep_hours, attendance_rate, study_consistency, tasks_on_time, assignments_on_time, revision_frequency, career_goal_clarity, daily_productivity, stress_level, age plus cohort_code with cross-fitted target encoding. Submission day is excluded because all three D.2 comparisons favored exclusion. Tree depth chosen by temporal CV: 2; Ridge alpha: 100; boosting trees selected on the chronological validation block: 68.

| Model | Train RMSE | Test RMSE | Test R² |
|---|---:|---:|---:|---|
| Baseline | 1.2685 | 1.2362 | -0.0029 |
| Linear Regression | 1.1252 | 1.1016 | 0.2036 |
| Ridge alpha=100 | 1.1297 | 1.1026 | 0.2022 |
| Decision Tree depth=2 | 1.1849 | 1.1458 | 0.1384 |
| Gradient Boosting (68 trees) | 1.0516 | 1.1050 | 0.1987 |

## Final discussion
1. Best generalization on the untouched test set: Linear Regression (RMSE 1.1016).
2. Largest generalization gap: Gradient Boosting (68 trees) (0.0534 CGPA points).
3. Limiting tree depth, selecting complexity with temporal validation, regularization, and leakage-safe preprocessing are the main controls; their effect is visible in the tables above.
4. Training error rewards fitting known examples and cannot measure performance on later unseen responses; validation compares candidate choices on data withheld from fitting.
5. Repeated test-guided choices gradually fit the test set too, turning it into another training signal and biasing the final estimate.
6. The best test R² is 0.204; modest explanatory power indicates the survey captures only part of the determinants of CGPA. A high score can also result from leakage or an unrealistic split, so evaluation design matters as much as the number.

Figures: part_a_cgpa_time.png, part_c_tree_depth.png, part_c_poly_degree.png.